In [1]:
import os
os.environ["PGPASSWORD"] = "aa8940aa"

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
from sqlalchemy import text

from db.connection import DatabaseConnection

# 日本語フォント（Windows: Yu Gothic / Meiryo, Mac: Hiragino, Linux: IPAexGothic）
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.dpi"] = 120
matplotlib.font_manager.fontManager.addfont(
    r"C:\Windows\Fonts\YuGothM.ttc"
)
plt.rcParams["font.family"] = "Yu Gothic"

engine = DatabaseConnection().get_engine()

def load_entries_with_results(engine, start, end):
    sql = text("""
        SELECT e.*, r.year, r.month_day, r.surface, r.field_size
        FROM raw.entries e
        JOIN raw.races r ON e.race_id = r.race_id
        WHERE (r.year || r.month_day)::int BETWEEN :start AND :end
        AND r.track_cd NOT BETWEEN 51 AND 59
        AND e.finish_pos > 0
        ORDER BY e.race_id, e.umaban
    """)
    df = pd.read_sql(sql, engine, params={"start": start, "end": end})
    df["race_date"] = pd.to_datetime(df["year"].astype(str) + df["month_day"], format="%Y%m%d")
    return df

print("Setup complete")

Setup complete


In [2]:
df_entries = load_entries_with_results(engine, "20180101", "20241231")
print(f"{df_entries['race_id'].nunique()} races, {len(df_entries)} entries")

42660 races, 516739 entries


In [3]:
yearly = df_entries.groupby(df_entries["race_date"].dt.year).agg(
    n_races=("race_id", "nunique"),
    n_entries=("race_id", "count"),
    avg_odds=("win_odds", "mean"),
    median_odds=("win_odds", "median"),
)
display(yearly)

,n_races,n_entries,avg_odds,median_odds
race_date,,,,
2018,6111,74506,453.064250,67.0
2019,6201,74627,390.637691,61.0
2020,6148,74694,395.031087,65.0
2021,6034,73582,391.134829,66.0
2022,6252,74917,415.599156,58.0
2023,6070,73409,446.707229,65.0
2024,5844,71004,463.672779,69.0


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_entries["win_odds"], bins=100, log=True, alpha=0.7, color="steelblue")
axes[0].set_xlabel("Win Odds")
axes[0].set_ylabel("Count (log)")
axes[0].set_title("Win Odds Distribution")
popularity_win_rate = df_entries.groupby("ninki").apply(
    lambda x: (x["finish_pos"] == 1).mean(), include_groups=False
)
popularity_win_rate.head(10).plot.bar(ax=axes[1], color="coral")
axes[1].set_ylabel("Win Rate")
axes[1].set_title("Win Rate by Popularity Rank")
plt.tight_layout()
plt.show()

In [5]:
for surface in ["turf", "dirt"]:
    sub = df_entries[df_entries["surface"] == surface]
    if len(sub) == 0:
        continue
    print(f"\n{surface}: {sub['race_id'].nunique()} races")
    print(f"  avg field size: {sub.groupby('race_id')['umaban'].count().mean():.1f}")
    fav_wr = (sub[sub['ninki']==1]['finish_pos']==1).mean()
    print(f"  1st fav win rate: {fav_wr:.3f}")


turf: 11977 races
  avg field size: 13.2
  1st fav win rate: 0.332

dirt: 30683 races
  avg field size: 11.7
  1st fav win rate: 0.421
